# 02 – Cache ảnh thật 256 px: Sketchy Extended & Flickr15K (Role A – Hạ tầng)

Theo **Mục 2.4 (Kế hoạch thực nghiệm 4 tuần v2 - chốt ngày 15/11)**:
- **Phạm vi:** CHỈ cache ảnh thật (photos) của Sketchy (73.002 ảnh) và Flickr15K (14.660 ảnh). Tổng cộng: **87.662 ảnh**.
- **Tuyệt đối KHÔNG cache sketch:** Sketch luôn đọc từ dữ liệu gốc (raw).
- **Quy tắc resize:** Cạnh NGẮN = 256 px (giữ tỉ lệ). Ảnh có cạnh ngắn <= 256 px thì giữ nguyên (KHÔNG phóng to).
- **Tối ưu:** Dùng `Image.draft()` giải nén nhanh + bộ lọc `LANCZOS`, lưu JPEG quality 95.
- **Đầu ra:** Thư mục `/kaggle/working/sbir-cache-256-photos` cùng 2 file metadata `cache_info.json` và `cache_manifest.csv`.

Thông số phiên chạy:
- **Accelerator:** CPU (không tốn quota GPU)
- **Internet:** On (để clone/pull repo)
- **Input datasets cần thêm:** `hongonthi/sketchyextended` và `hongonthi/flickr15k`

In [ ]:
SKETCHY_ROOT = "/kaggle/input/datasets/hongonthi/sketchyextended/sketchy"
FLICKR_ROOT  = "/kaggle/input/datasets/hongonthi/flickr15k/Flickr15K"
OUT_DIR      = "/kaggle/working/sbir-cache-256-photos"
REPO         = "https://github.com/thaihoang104569/CS406.git"

import os, sys, json
import pandas as pd
WORK = "/kaggle/working/CS406"
if os.path.isdir(WORK):
    !git -C {WORK} pull -q
else:
    !git clone -q {REPO} {WORK}

!git -C {WORK} log --oneline -1

## 1. Chạy thử nghiệm nhanh (--debug: 200 ảnh)

In [ ]:
!python {WORK}/scripts/cache_images.py \
    --sketchy_root "{SKETCHY_ROOT}" \
    --flickr_root  "{FLICKR_ROOT}" \
    --out_dir      "{OUT_DIR}" \
    --workers      4 \
    --debug

## 2. Chạy toàn bộ dữ liệu (~15–25 phút trên CPU Kaggle 4 nhân)

In [ ]:
!python {WORK}/scripts/cache_images.py \
    --sketchy_root "{SKETCHY_ROOT}" \
    --flickr_root  "{FLICKR_ROOT}" \
    --out_dir      "{OUT_DIR}" \
    --workers      4

## 3. Kiểm tra nghiệm thu (Acceptance Check theo Mục 2.4)

In [ ]:
info_file = f"{OUT_DIR}/cache_info.json"
manifest_file = f"{OUT_DIR}/cache_manifest.csv"

with open(info_file, "r", encoding="utf-8") as f:
    info = json.load(f)
print("=== CACHE INFO ===")
print(json.dumps(info, indent=2))

df_manifest = pd.read_csv(manifest_file)
print(f"\n=== MANIFEST CHECK ===")
print(f"Tổng số ảnh thật trong manifest: {len(df_manifest):,}")
print(df_manifest["dataset"].value_counts())

# Điều kiện chấp nhận 1: Số ảnh thật khớp với dữ liệu split
EXPECTED_SKETCHY_PHOTOS = 73002
EXPECTED_FLICKR_PHOTOS = 14660
assert len(df_manifest) == EXPECTED_SKETCHY_PHOTOS + EXPECTED_FLICKR_PHOTOS, (
    f"Lỗi: Số lượng ảnh manifest ({len(df_manifest)}) không khớp kỳ vọng "
    f"({EXPECTED_SKETCHY_PHOTOS + EXPECTED_FLICKR_PHOTOS})"
)
print("\n✅ NGHIỆM THU ĐẠT: Số lượng ảnh manifest khớp 100% với file split CSV!")

# Kiểm tra dung lượng thư mục
!du -sh {OUT_DIR}